# Machine Learning Notebook: COMP3920SEF
## Introduction
This notebook is designed to make a images classification using tensorflow
We will go through the following steps:
1. Import necessary libraries
2. Load and explore the dataset
3. Preprocess data
4. Train a machine learning model
5. Evaluate the model
6. Make predictions

In [13]:
# Importing essential libraries for machine learning and image processing

import matplotlib.pyplot as plt  # Used for visualizing data through plots and graphs
import numpy as np  # Provides support for numerical computations and arrays
import tensorflow as tf  # TensorFlow, an open-source machine learning framework
import PIL  # Pillow, a Python Imaging Library for handling image files
import pathlib  # Helps manage file paths and directories

# TensorFlow's Keras module for building deep learning models
from tensorflow import keras

# Layers module provides various types of layers (e.g., Dense, Conv2D, MaxPooling2D)
from tensorflow.keras import layers

# Sequential API to create layer-by-layer deep learning models
from tensorflow.keras.models import Sequential

# ImageDataGenerator is used for image preprocessing and data augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator


#Set up

##Train Set

In [14]:
# Set the Path for the dataset
data_dir = pathlib.Path("/content/drive/MyDrive/Nike_Adidas_converse_Shoes_image_dataset/train/")

# Image size
image_size = (240,240)

# Batch size
batch_size = 32

# normalize the image from 255 pixel to [0,1]
# 8 bit for 255
datagen = ImageDataGenerator(rescale= 1./255)

# Dataloader
train_data_gen = datagen.flow_from_directory(
    data_dir,
    target_size = image_size,
    batch_size = batch_size,
    class_mode = 'categorical',
    subset = None
)

Found 603 images belonging to 3 classes.


## Validate Set

In [15]:
data_dir = pathlib.Path('/content/drive/MyDrive/Nike_Adidas_converse_Shoes_image_dataset/validate/')
image_size = (240,240)
batch_size = 32
datagen = ImageDataGenerator(rescale=1./255)
validate_data_gen = datagen.flow_from_directory(
    data_dir,
    target_size = image_size,
    batch_size = batch_size,
    class_mode = 'categorical',
    subset = None
)

Found 90 images belonging to 3 classes.


## Check the train Data Info

In [16]:
# you can check your labels (class indices) and number of samples before you start to train the model
import numpy as np
print("Class indices:", train_data_gen.class_indices)
print("Total number of samples:", train_data_gen.samples)
class_counts = {label: np.sum(train_data_gen.labels == label) for label in range(len(train_data_gen.class_indices))}
print("Number of images per class:", class_counts)

Class indices: {'adidas': 0, 'converse': 1, 'nike': 2}
Total number of samples: 603
Number of images per class: {0: np.int64(201), 1: np.int64(201), 2: np.int64(201)}


In [17]:
# you can check your input and output shape here
batch_images, batch_labels = next(train_data_gen)
print("Shape of batch_images (input):", batch_images.shape)  # (batch_size, image_size1, image_size2, channels)
print("Shape of batch_labels (output):", batch_labels.shape)  # (batch_size, num_classes)
print("First 5 labels in batch:\n", batch_labels[:5])

Shape of batch_images (input): (32, 240, 240, 3)
Shape of batch_labels (output): (32, 3)
First 5 labels in batch:
 [[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


## Set up the CNNs Layers

## Manual CNNs

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Defined the CNNs model
model = Sequential()

# Layers
## First layer is a Convulational 2D with batch 32 and 3x3
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(240, 240, 3)))  # 3 refer to RGB
## Max Polling with 2x2
model.add(MaxPooling2D((2, 2)))
## Second layer of conv2D of 64 filters
model.add(Conv2D(64, (3, 3), activation='relu'))
## Max Polling with 2x2
model.add(MaxPooling2D((2, 2)))
## Covert into a 1D vector
model.add(Flatten())
## Added 128 neurons
model.add(Dense(128, activation='relu'))
## Drop 50% to avoid overfitting
model.add(Dropout(0.5))
## Output 3 values of Nike, Adidas, or Converse.
model.add(Dense(3, activation='softmax'))

model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()
history = model.fit(
    train_data_gen,
    epochs=20,
    validation_data=validate_data_gen
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 238, 238, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 119, 119, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 117, 117, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 58, 58, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 215296)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    27,558,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,577,795 (105.20 MB)

 Trainable params: 27,577,795 (105.20 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


KeyboardInterrupt: 

## Use Pre-Trained model

In [20]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Load MobileNetV2 with pre-trained ImageNet weights (without top classification layer)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(240, 240, 3))

# Freeze initial Layers since the pre-train model already trained
for layer in base_model.layers:
    layer.trainable = False

# Create new classification layers
model = Sequential([
    base_model, # Attach pre-trained model
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # Output for 3 shoe brands: Nike, Adidas, Converse
])


model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train only the new classification layers
history = model.fit(train_data_gen, epochs=20, validation_data=validate_data_gen)


<ipython-input-20-3e488ab9dddb>:7: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(240, 240, 3))


Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.3886 - loss: 11.1900 - val_accuracy: 0.6444 - val_loss: 1.0643
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.7286 - loss: 0.6275 - val_accuracy: 0.6444 - val_loss: 0.7959
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.7813 - loss: 0.5029 - val_accuracy: 0.7778 - val_loss: 0.6698
Epoch 4/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.8267 - loss: 0.3776 - val_accuracy: 0.8222 - val_loss: 0.7167
Epoch 5/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.8755 - loss: 0.3338 - val_accuracy: 0.8000 - val_loss: 0.6766
Epoch 6/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8890 - loss: 0.2737 - val_accuracy: 0.7333 - val_loss: 0.7195
Epoch 7/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8983 - loss: 0.2383 - val_accuracy: 0.7333 - val_loss: 0.6969
Epoch 8/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.8726 - loss: 0.2443 - val_accuracy: 0.8000 - val_loss

## Save the result

In [21]:
model.save('CNNs.h5')

##Load the model

In [22]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pathlib


model = load_model('CNNs.h5') # load model


data_dir = pathlib.Path('/content/drive/MyDrive/Nike_Adidas_converse_Shoes_image_dataset/validate/') #set the path of the dataset
image_size = (240, 240)
batch_size = 32

# create dataloader as the input for the model
datagen = ImageDataGenerator(rescale=1./255)

validate_data_gen = datagen.flow_from_directory(
    data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# evaluate model
loss, accuracy = model.evaluate(validate_data_gen)
print(f"\nValidation Accuracy: {accuracy*100:.2f}%")

Found 90 images belonging to 3 classes.
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.8603 - loss: 0.7209

Validation Accuracy: 82.22%
